# PDF Analyzer: RAG and Retrieval Evaluation

This notebook combines PDF loading, chunking, vector search, retrieval testing, and optional answer generation in one workflow. It is designed to work with any text-based PDF. Scanned/image-only PDFs need OCR before loading.

Run the setup and indexing cells first. Retrieval and evaluation do not need an LLM API key. Edit the ground-truth chunk IDs after reviewing the chunk table before interpreting evaluation metrics.


In [ ]:
# If needed, install dependencies in this notebook's environment:
# %pip install langchain-community langchain-text-splitters langchain-huggingface langchain-google-genai sentence-transformers faiss-cpu pypdf


## 1. Imports and PDF selection


In [ ]:
import os
import re
from pathlib import Path
from typing import Iterable

from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = Path("resume.pdf")

if not PDF_PATH.is_absolute() and not PDF_PATH.is_file():
    workspace_path = Path.cwd() / "WEEK2" / "DAY9" / "RAG" / PDF_PATH
    if workspace_path.is_file():
        PDF_PATH = workspace_path

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH.resolve()}. Set PDF_PATH to the PDF you want to analyze."
    )


## 2. Load all PDF pages and preserve page metadata


In [ ]:
pdf_pages = PyPDFLoader(str(PDF_PATH)).load()
if not pdf_pages:
    raise ValueError(f"No pages were loaded from {PDF_PATH}.")

source_path = str(PDF_PATH.resolve())
page_documents = []
for page in pdf_pages:
    page_text = re.sub(r"\s+", " ", page.page_content).strip()
    if page_text:
        metadata = {**page.metadata, "source": source_path}
        page_documents.append(Document(page_content=page_text, metadata=metadata))

if not page_documents:
    raise ValueError(
        "No extractable text was found in the PDF. If it is scanned, run OCR first."
    )

print(f"Loaded {len(pdf_pages)} PDF pages; {len(page_documents)} contain text.")
display(
    [
        {
            "page": int(doc.metadata.get("page", i)) + 1,
            "characters": len(doc.page_content),
            "preview": doc.page_content[:180],
        }
        for i, doc in enumerate(page_documents)
    ]
)


## 3. Split into chunks and remove exact duplicates

Only whitespace is normalized. Keep original words, punctuation, and stop words for stronger semantic retrieval. Duplicate chunks retain the page numbers where they appeared.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    add_start_index=True,
)
split_documents = splitter.split_documents(page_documents)

unique_by_text = {}
for document in split_documents:
    key = re.sub(r"\s+", " ", document.page_content).strip().casefold()
    if not key:
        continue

    page = document.metadata.get("page")
    if key not in unique_by_text:
        metadata = {**document.metadata, "pages": []}
        if page is not None:
            metadata["pages"].append(int(page))
        unique_by_text[key] = Document(
            page_content=document.page_content,
            metadata=metadata,
        )
    elif page is not None:
        pages = unique_by_text[key].metadata["pages"]
        if int(page) not in pages:
            pages.append(int(page))

chunks = list(unique_by_text.values())
for chunk_number, document in enumerate(chunks):
    document.metadata["chunk_id"] = f"chunk-{chunk_number:04d}"

if not chunks:
    raise ValueError("No non-empty chunks were created from the PDF.")

print(f"Created {len(split_documents)} chunks; {len(chunks)} remain after deduplication.")


## 4. Review chunk IDs before retrieval evaluation


In [ ]:
chunk_table = [
    {
        "chunk_id": document.metadata["chunk_id"],
        "pages": [page + 1 for page in document.metadata.get("pages", [])],
        "characters": len(document.page_content),
        "text": document.page_content,
    }
    for document in chunks
]
display(chunk_table)


## 5. Embed chunks and build the FAISS index


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
)
print(f"FAISS index ready with {len(chunks)} chunks.")


## 6. Retrieve relevant chunks


In [ ]:
def retrieve(query: str, k: int = 4) -> list[Document]:
    if not query or not query.strip():
        raise ValueError("The retrieval query must not be empty.")
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    return vector_store.similarity_search(query.strip(), k=min(k, len(chunks)))


def retrieved_chunk_ids(documents: Iterable[Document]) -> list[str]:
    return [document.metadata["chunk_id"] for document in documents]


In [ ]:
demo_query = "Summarize the document's key information."
demo_results = retrieve(demo_query, k=4)
print(f"Query: {demo_query}")
for rank, document in enumerate(demo_results, start=1):
    page_numbers = [page + 1 for page in document.metadata.get("pages", [])]
    print(
        f"\n{rank}. {document.metadata['chunk_id']} "
        f"(page(s): {page_numbers or 'unknown'})"
    )
    print(document.page_content)


## 7. Retrieval metrics: Recall@k, Precision@k, and MRR


In [ ]:
def recall_at_k(retrieved: list[str], relevant: list[str], k: int) -> float | None:
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    relevant_ids = set(relevant)
    if not relevant_ids:
        return None
    return len(set(retrieved[:k]) & relevant_ids) / len(relevant_ids)


def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    top_k = retrieved[:k]
    if not top_k:
        return 0.0
    return len(set(top_k) & set(relevant)) / len(top_k)


def reciprocal_rank(retrieved: list[str], relevant: list[str]) -> float:
    relevant_ids = set(relevant)
    if not relevant_ids:
        return 0.0
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank
    return 0.0


## 8. Curate ground truth and evaluate retrieval

For each query, use the chunk table above to identify **all chunks that truly answer it**, then enter their `chunk_id` values below. Empty lists are intentionally not treated as ground truth. Keyword overlap alone is not proof that a chunk is relevant.


In [ ]:
evaluation_queries = [
    "What is the document about?",
    "What are the most important qualifications or findings?",
]

# Replace [] with the manually reviewed chunk IDs relevant to each query.
ground_truth = {
    "What is the document about?": [],
    "What are the most important qualifications or findings?": [],
}

K = 4
if K <= 0:
    raise ValueError("K must be a positive integer.")

evaluation_rows = []
for query in evaluation_queries:
    relevant_ids = ground_truth.get(query, [])
    if not relevant_ids:
        print(f"Skipping '{query}': curate its ground_truth chunk IDs first.")
        continue

    found_ids = retrieved_chunk_ids(retrieve(query, k=K))
    effective_k = min(K, len(chunks))
    row = {
        "query": query,
        "relevant_chunk_ids": relevant_ids,
        "retrieved_chunk_ids": found_ids,
        "recall_at_k": recall_at_k(found_ids, relevant_ids, effective_k),
        "precision_at_k": precision_at_k(found_ids, relevant_ids, effective_k),
        "reciprocal_rank": reciprocal_rank(found_ids, relevant_ids),
    }
    evaluation_rows.append(row)

if evaluation_rows:
    display(evaluation_rows)
    mean_recall = sum(row["recall_at_k"] for row in evaluation_rows) / len(evaluation_rows)
    mean_precision = sum(row["precision_at_k"] for row in evaluation_rows) / len(evaluation_rows)
    mean_reciprocal_rank = (
        sum(row["reciprocal_rank"] for row in evaluation_rows) / len(evaluation_rows)
    )
    print(f"Evaluated queries: {len(evaluation_rows)}")
    print(f"Mean Recall@{K}: {mean_recall:.3f}")
    print(f"Mean Precision@{K}: {mean_precision:.3f}")
    print(f"Mean Reciprocal Rank: {mean_reciprocal_rank:.3f}")
else:
    print("No evaluation metrics yet. Add reviewed chunk IDs to ground_truth and rerun.")


## 9. Optional Gemini answer generation (not called automatically)


In [ ]:
def answer_question(query: str, k: int = 4) -> str:
    api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GeminiAPIKey")
    if not api_key:
        raise RuntimeError(
            "Set GOOGLE_API_KEY or GeminiAPIKey in the environment before generation."
        )

    documents = retrieve(query, k=k)
    if not documents:
        raise ValueError("No chunks were retrieved; unable to answer this question.")

    context_parts = []
    for document in documents:
        pages = [page + 1 for page in document.metadata.get("pages", [])]
        page_label = ", ".join(str(page) for page in pages) if pages else "unknown"
        context_parts.append(
            f"[{document.metadata['chunk_id']}; page(s) {page_label}]\n"
            f"{document.page_content}"
        )
    context = "\n\n".join(context_parts)

    from langchain_google_genai import ChatGoogleGenerativeAI

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Answer only using the supplied PDF context. If it does not contain "
                "enough information, say so. Cite supporting page number(s) from "
                "the context in your answer. Do not invent facts or citations.",
            ),
            ("human", "Question: {question}\n\nPDF context:\n{context}"),
        ]
    )
    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=api_key,
    )
    chain = prompt | model | StrOutputParser()
    return chain.invoke({"question": query, "context": context})


# Optional: uncomment to make a Gemini API call after setting an API key.
# answer = answer_question("What is the document about?")
# print(answer)
